# ViLBERT-Optimized: Multimodal Sentiment Analysis on MVSA-Single

**Architecture**: BERT (text) + ViT-B/16 (image) + 3-layer Cross-Attention + Gated Fusion

**Optimizations**: EMA, R-Drop, Label Smoothing, AMP, Gradient Accumulation, Cosine Warmup

In [2]:
# =====================================================
# Cell 1 - Mount Google Drive
# =====================================================

import os



dataset_path = 'D:\/MVSA_SINGLE'
if os.path.exists(dataset_path):
    print(f'Drive mounted. Dataset path: {dataset_path}')
    print(f'Contents: {os.listdir(dataset_path)[:8]}')
else:
    print(f'Path not found: {dataset_path}')

Drive mounted. Dataset path: D:\/MVSA_SINGLE
Contents: ['bert_lstm_checkpoints', 'bert_lstm_final', 'data', 'labelResultAll.txt', 'mvsa_single_processed_224.parquet', 'vit_lstm_checkpoints', 'vit_lstm_final']


In [3]:
# =====================================================
# Cell - Check Parquet Dataset
# =====================================================
import pandas as pd
import os
import sys
import warnings

try:
    parquet_path = CONFIG.get('parquet_path', 'D:\/MVSA_SINGLE/mvsa_single_processed_224.parquet')
except NameError:
    parquet_path = 'D:\/MVSA_SINGLE/mvsa_single_processed_224.parquet'

if not os.path.exists(parquet_path):
    msg = f"Parquet file {parquet_path} not found. Please run 'mvsa-to-parquet.ipynb' first to generate the dataset."
    warnings.warn(msg, UserWarning)
    sys.exit(f"FileNotFoundError: {msg}")
else:
    print(f"File parquet found, loading from: {parquet_path}")
    # Optional preview load
    df = pd.read_parquet(parquet_path)
    print(f"Total entries loaded: {len(df)}")
    if 'final_sentiment' in df.columns:
        print("\nSentiment Distribution:")
        print(df['final_sentiment'].value_counts())
    display(df.head())


File parquet found, loading from: D:\/MVSA_SINGLE/mvsa_single_processed_224.parquet
Total entries loaded: 4511

Sentiment Distribution:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."


In [4]:
# =====================================================
# Cell 3 - Install & Import Dependencies
# =====================================================
%pip install transformers accelerate scikit-learn pyarrow torchvision timm -q

import copy, io, json, math, os, random, time, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import timm
import torchvision.transforms as transforms

from transformers import BertModel, BertTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True

print('All libraries imported successfully.')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

Note: you may need to restart the kernel to use updated packages.


c:\Users\Residensi ADW\anaconda3\envs\env_gpu_pytorch_erl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.
PyTorch  : 2.5.1
CUDA     : True
GPU      : NVIDIA GeForce RTX 3050


In [5]:
# =====================================================
# Cell 4 - Optimized Configuration
# =====================================================
CONFIG = {
    # -- Paths ------------------------------------------------
    "parquet_path"   : "D:\/MVSA_SINGLE/mvsa_single_processed_224.parquet",
    "checkpoint_dir" : "D:\/MVSA_SINGLE/vilbert_opt_checkpoints",
    "save_dir"       : "D:\/MVSA_SINGLE/vilbert_opt_final",

    # -- Text Encoder (BERT) ----------------------------------
    "bert_model_name"  : "bert-base-uncased",
    "bert_hidden_size" : 768,
    "max_text_length"  : 128,

    # -- Image Encoder (ViT-B/16) -----------------------------
    "image_backbone"        : "vit_base_patch16_224",
    "image_feature_dim"     : 768,
    "image_input_size"      : 224,
    "image_drop_cls_token"  : True,     # use patch tokens only for richer spatial info
    "image_trainable_blocks": 4,        # unfreeze last 4 ViT blocks
    "image_dropout"         : 0.10,

    # -- Co-Attention -----------------------------------------
    "co_attn_layers"  : 3,              # 3 layers for deeper cross-modal reasoning
    "co_attn_heads"   : 8,
    "co_attn_dropout" : 0.10,
    "ffn_hidden_dim"  : 3072,           # 4x hidden_dim (standard transformer ratio)

    # -- Fusion & Classifier ----------------------------------
    "fusion_strategy"    : "cls-image-absdiff-gated",
    "fusion_input_dim"   : 768 * 3,     # [CLS; img_pool; |CLS - img_pool|]
    "classifier_hidden"  : 768,
    "num_classes"        : 3,
    "classifier_dropout" : 0.25,

    # -- Training Hyperparameters -----------------------------
    "epochs"              : 30,         # more epochs for exhaustive search w/ early stopping
    "batch_size"          : 16,
    "grad_accum_steps"    : 2,          # effective batch = 32
    "bert_lr"             : 2e-5,
    "vision_lr"           : 1e-5,       # gentle for pretrained ViT
    "new_params_lr"       : 1e-4,
    "weight_decay"        : 0.01,
    "warmup_ratio"        : 0.10,       # 10% linear warmup
    "gradient_clip"       : 1.0,
    "early_stop_patience" : 7,          # more patience for thorough convergence
    "label_smoothing"     : 0.05,
    "loss_name"           : "weighted_ce",
    "use_amp"             : True,

    # -- EMA (Exponential Moving Average) ---------------------
    "use_ema"    : True,
    "ema_decay"  : 0.999,

    # -- R-Drop Regularization --------------------------------
    "use_rdrop"     : True,
    "rdrop_alpha"   : 1.0,              # weight for KL divergence term

    # -- Data Split -------------------------------------------
    "train_ratio" : 0.80,
    "val_ratio"   : 0.10,
    "test_ratio"  : 0.10,

    # -- Reproducibility --------------------------------------
    "seed"        : 42,
    "num_workers"  : 0,  # Windows fix: DataLoader multiprocessing causes bottleneck
    "device"       : "cuda" if torch.cuda.is_available() else "cpu",
}

# ── Seed everything ──────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["save_dir"], exist_ok=True)

# ── Pretty-print config ──────────────────────────────────
def print_config(cfg):
    SEP = '=' * 64
    LINE = '-' * 64
    print(SEP)
    print('  ViLBERT-Optimized  |  Configuration Summary')
    print(SEP)

    sections = {
        'Device': [
            ('Device', cfg['device'].upper()),
            ('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'),
        ],
        'Text Encoder': [
            ('Model', cfg['bert_model_name']),
            ('Hidden dim', cfg['bert_hidden_size']),
            ('Max tokens', cfg['max_text_length']),
        ],
        'Image Encoder': [
            ('Backbone', cfg['image_backbone']),
            ('Feature dim', cfg['image_feature_dim']),
            ('Drop CLS token', cfg['image_drop_cls_token']),
            ('Trainable blocks', cfg['image_trainable_blocks']),
            ('Input size', f"{cfg['image_input_size']}x{cfg['image_input_size']}"),
        ],
        'Co-Attention': [
            ('Layers', cfg['co_attn_layers']),
            ('Heads', cfg['co_attn_heads']),
            ('FFN hidden dim', cfg['ffn_hidden_dim']),
            ('Dropout', cfg['co_attn_dropout']),
        ],
        'Fusion & Classifier': [
            ('Fusion strategy', cfg['fusion_strategy']),
            ('Fusion input dim', cfg['fusion_input_dim']),
            ('Classifier hidden', cfg['classifier_hidden']),
            ('Classifier dropout', cfg['classifier_dropout']),
            ('Num classes', cfg['num_classes']),
        ],
        'Training': [
            ('Epochs', cfg['epochs']),
            ('Batch size (real)', cfg['batch_size']),
            ('Grad accum steps', cfg['grad_accum_steps']),
            ('Effective batch', cfg['batch_size'] * cfg['grad_accum_steps']),
            ('BERT LR', cfg['bert_lr']),
            ('Vision LR', cfg['vision_lr']),
            ('New params LR', cfg['new_params_lr']),
            ('Weight decay', cfg['weight_decay']),
            ('Warmup ratio', f"{cfg['warmup_ratio']} ({int(cfg['warmup_ratio']*100)}%)"),
            ('Gradient clip', cfg['gradient_clip']),
            ('Early stop patience', cfg['early_stop_patience']),
            ('Label smoothing', cfg['label_smoothing']),
            ('Loss function', cfg['loss_name']),
            ('Mixed precision (AMP)', cfg['use_amp']),
        ],
        'Regularization': [
            ('EMA enabled', cfg['use_ema']),
            ('EMA decay', cfg['ema_decay']),
            ('R-Drop enabled', cfg['use_rdrop']),
            ('R-Drop alpha', cfg['rdrop_alpha']),
        ],
        'Data': [
            ('Split (train/val/test)', f"{cfg['train_ratio']}/{cfg['val_ratio']}/{cfg['test_ratio']}"),
            ('Random seed', cfg['seed']),
        ],
    }

    for section_name, items in sections.items():
        print(f'\n  [{section_name}]')
        for key, val in items:
            print(f'    {key:<22}: {val}')

    print(f'\n{SEP}')

print_config(CONFIG)

  ViLBERT-Optimized  |  Configuration Summary

  [Device]
    Device                : CUDA
    GPU                   : NVIDIA GeForce RTX 3050

  [Text Encoder]
    Model                 : bert-base-uncased
    Hidden dim            : 768
    Max tokens            : 128

  [Image Encoder]
    Backbone              : vit_base_patch16_224
    Feature dim           : 768
    Drop CLS token        : True
    Trainable blocks      : 4
    Input size            : 224x224

  [Co-Attention]
    Layers                : 3
    Heads                 : 8
    FFN hidden dim        : 3072
    Dropout               : 0.1

  [Fusion & Classifier]
    Fusion strategy       : cls-image-absdiff-gated
    Fusion input dim      : 2304
    Classifier hidden     : 768
    Classifier dropout    : 0.25
    Num classes           : 3

  [Training]
    Epochs                : 30
    Batch size (real)     : 16
    Grad accum steps      : 2
    Effective batch       : 32
    BERT LR               : 2e-05
    Vision 

In [6]:
# =====================================================
# Cell 5 - Prepare Data Splits
# =====================================================
print(f"Loading parquet from: {CONFIG['parquet_path']}")
df_full = pd.read_parquet(CONFIG["parquet_path"])
print(f"Total rows loaded: {len(df_full)}")

# Drop NaN
before = len(df_full)
df_full = df_full.dropna(subset=["text_content", "image_bytes", "final_sentiment"]).reset_index(drop=True)
print(f"Rows after dropna: {len(df_full)}  (dropped {before - len(df_full)})")

# Encode labels
LABEL_MAP  = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_IMAP = {v: k for k, v in LABEL_MAP.items()}
df_full["label"] = df_full["final_sentiment"].str.strip().map(LABEL_MAP)
missing = df_full["label"].isna().sum()
if missing > 0:
    print(f"WARN: {missing} rows have unrecognised labels -- dropping.")
    df_full = df_full.dropna(subset=["label"]).reset_index(drop=True)
df_full["label"] = df_full["label"].astype(int)

print(f"\nLabel distribution:")
for lbl, idx in LABEL_MAP.items():
    count = (df_full["label"] == idx).sum()
    print(f"  {idx}  {lbl:<10}  {count:>5}  ({count/len(df_full)*100:.1f}%)")

# Stratified split: 80 / 10 / 10
df_train, df_tmp = train_test_split(
    df_full,
    test_size=(CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_full["label"],
    random_state=CONFIG["seed"],
)
df_val, df_test = train_test_split(
    df_tmp,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_tmp["label"],
    random_state=CONFIG["seed"],
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"\nData split (stratified):")
print(f"  Train : {len(df_train):>5} rows")
print(f"  Val   : {len(df_val):>5} rows")
print(f"  Test  : {len(df_test):>5} rows")
print(f"  Total : {len(df_train)+len(df_val)+len(df_test):>5} rows")

Loading parquet from: D:\/MVSA_SINGLE/mvsa_single_processed_224.parquet
Total rows loaded: 4511
Rows after dropna: 4511  (dropped 0)

Label distribution:
  0  negative     1358  (30.1%)
  1  neutral       470  (10.4%)
  2  positive     2683  (59.5%)

Data split (stratified):
  Train :  3608 rows
  Val   :   451 rows
  Test  :   452 rows
  Total :  4511 rows


In [7]:
# =====================================================
# Cell 6 - Dataset & DataLoaders
# =====================================================
tokenizer = BertTokenizer.from_pretrained(CONFIG["bert_model_name"])

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG["image_input_size"], scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05)
    ], p=0.4),
    transforms.RandomAffine(degrees=8, translate=(0.05, 0.05)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CONFIG["image_input_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])


class MVSADataset(Dataset):
    def __init__(self, dataframe, tokenizer, image_transform, max_length):
        self.df        = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = image_transform
        self.max_len   = max_length

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _decode_image(img_bytes):
        if isinstance(img_bytes, (bytes, bytearray)):
            raw = bytes(img_bytes)
        elif isinstance(img_bytes, (memoryview, np.ndarray)):
            raw = img_bytes.tobytes()
        else:
            raw = bytes(img_bytes)
        return Image.open(io.BytesIO(raw)).convert("RGB")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = self._decode_image(row["image_bytes"])
        except Exception:
            image = Image.new("RGB", (CONFIG["image_input_size"], CONFIG["image_input_size"]), (0,0,0))

        image_tensor = self.transform(image)
        enc = self.tokenizer(
            str(row["text_content"]),
            padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "token_type_ids" : enc["token_type_ids"].squeeze(0),
            "image"          : image_tensor,
            "label"          : torch.tensor(row["label"], dtype=torch.long),
        }


loader_kwargs = {
    "batch_size"  : CONFIG["batch_size"],
    "num_workers"  : CONFIG["num_workers"],
    "pin_memory"   : torch.cuda.is_available(),
}
if CONFIG["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True

train_dataset = MVSADataset(df_train, tokenizer, train_transform, CONFIG["max_text_length"])
val_dataset   = MVSADataset(df_val,   tokenizer, eval_transform,  CONFIG["max_text_length"])
test_dataset  = MVSADataset(df_test,  tokenizer, eval_transform,  CONFIG["max_text_length"])

train_loader = DataLoader(train_dataset, shuffle=True,  drop_last=True, **loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kwargs)

print(f"Train batches: {len(train_loader)}  ({len(train_dataset)} samples)")
print(f"Val   batches: {len(val_loader)}  ({len(val_dataset)} samples)")
print(f"Test  batches: {len(test_loader)}  ({len(test_dataset)} samples)")

batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_ids      : {batch['input_ids'].shape}")
print(f"  attention_mask : {batch['attention_mask'].shape}")
print(f"  image          : {batch['image'].shape}")
print(f"  label          : {batch['label'].shape}  vals={batch['label'].tolist()[:8]}")

Train batches: 225  (3608 samples)
Val   batches: 29  (451 samples)
Test  batches: 29  (452 samples)

Batch shapes:
  input_ids      : torch.Size([16, 128])
  attention_mask : torch.Size([16, 128])
  image          : torch.Size([16, 3, 224, 224])
  label          : torch.Size([16])  vals=[2, 2, 2, 2, 0, 2, 0, 0]


In [8]:
# =====================================================
# Cell 7 - Model Architecture
# =====================================================

# --- Learned Attention Pooling ---
class LearnedAttentionPool(nn.Module):
    """Weighted-sum pooling with learnable attention scores."""
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, hidden_states, mask=None):
        scores = self.score(hidden_states).squeeze(-1)  # [B, L]
        if mask is not None:
            scores = scores.masked_fill(~mask, -1e4)
        weights = torch.softmax(scores, dim=-1)
        return torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)  # [B, D]


# --- Co-Attention Layer (Pre-LN variant for stability) ---
class CoAttentionLayer(nn.Module):
    """Bidirectional cross-attention: text<->image with Pre-LN and residual."""
    def __init__(self, hidden_dim, num_heads, ffn_dim, dropout):
        super().__init__()
        # Text attending to Image
        self.t_norm1 = nn.LayerNorm(hidden_dim)
        self.t2v_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.t_norm2 = nn.LayerNorm(hidden_dim)
        self.t_ffn = nn.Sequential(
            nn.Linear(hidden_dim, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, hidden_dim), nn.Dropout(dropout),
        )
        # Image attending to Text
        self.v_norm1 = nn.LayerNorm(hidden_dim)
        self.v2t_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.v_norm2 = nn.LayerNorm(hidden_dim)
        self.v_ffn = nn.Sequential(
            nn.Linear(hidden_dim, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, hidden_dim), nn.Dropout(dropout),
        )

    def forward(self, text_feats, image_feats, text_key_padding_mask=None):
        # Fix all-masked rows to prevent NaN
        if text_key_padding_mask is not None:
            all_masked = text_key_padding_mask.all(dim=1)
            if all_masked.any():
                text_key_padding_mask = text_key_padding_mask.clone()
                text_key_padding_mask[all_masked] = False

        # Pre-LN: text attending to image (query=text, key/value=image)
        t_normed = self.t_norm1(text_feats)
        t_attn, _ = self.t2v_attn(query=t_normed, key=image_feats, value=image_feats)
        text_out = text_feats + t_attn                    # residual
        text_out = text_out + self.t_ffn(self.t_norm2(text_out))  # FFN + residual

        # Pre-LN: image attending to text (query=image, key/value=text)
        v_normed = self.v_norm1(image_feats)
        v_attn, _ = self.v2t_attn(query=v_normed, key=text_feats, value=text_feats,
                                   key_padding_mask=text_key_padding_mask)
        image_out = image_feats + v_attn                  # residual
        image_out = image_out + self.v_ffn(self.v_norm2(image_out))  # FFN + residual

        return text_out, image_out


# --- ViT Image Encoder ---
class ViTImageEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = timm.create_model(
            config["image_backbone"], pretrained=True,
            num_classes=0, global_pool="",
        )
        self.drop_cls = config["image_drop_cls_token"]
        self.dropout  = nn.Dropout(config["image_dropout"])
        self._freeze(config["image_trainable_blocks"])

    def _freeze(self, trainable_blocks):
        for p in self.backbone.parameters():
            p.requires_grad = False
        # Unfreeze last N blocks
        if hasattr(self.backbone, "blocks"):
            for block in self.backbone.blocks[-trainable_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
        # Always unfreeze norm, patch_embed, cls_token, pos_embed
        for attr in ["norm", "fc_norm", "head_norm", "patch_embed"]:
            mod = getattr(self.backbone, attr, None)
            if mod is not None:
                for p in mod.parameters():
                    p.requires_grad = True
        for attr in ["cls_token", "pos_embed"]:
            val = getattr(self.backbone, attr, None)
            if isinstance(val, nn.Parameter):
                val.requires_grad = True

    def forward(self, pixel_values):
        tokens = self.backbone.forward_features(pixel_values)
        if isinstance(tokens, (list, tuple)):
            tokens = tokens[-1]
        if tokens.dim() == 4:
            tokens = tokens.flatten(2).transpose(1, 2)
        if self.drop_cls and tokens.size(1) > 1:
            tokens = tokens[:, 1:, :]   # drop CLS, keep patches
        return self.dropout(tokens)


# --- ViLBERT-Optimized ---
class ViLBERTOptimized(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config["bert_hidden_size"]

        # Encoders
        self.bert          = BertModel.from_pretrained(config["bert_model_name"])
        self.image_encoder = ViTImageEncoder(config)

        # Norms after encoders
        self.text_norm  = nn.LayerNorm(d)
        self.image_norm = nn.LayerNorm(d)

        # Co-Attention stack
        self.co_attn_layers = nn.ModuleList([
            CoAttentionLayer(d, config["co_attn_heads"], config["ffn_hidden_dim"], config["co_attn_dropout"])
            for _ in range(config["co_attn_layers"])
        ])

        # Pooling
        self.text_pool  = LearnedAttentionPool(d, config["classifier_dropout"])
        self.image_pool = LearnedAttentionPool(d, config["classifier_dropout"])

        # Gated fusion: [text_pool, image_pool, |text_pool - image_pool|]
        fusion_dim = config["fusion_input_dim"]
        self.fusion_gate = nn.Sequential(
            nn.Linear(fusion_dim, d), nn.GELU(), nn.Dropout(config["classifier_dropout"]),
            nn.Linear(d, fusion_dim), nn.Sigmoid(),
        )

        # Classifier with residual bottleneck
        ch = config["classifier_hidden"]
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, ch),
            nn.LayerNorm(ch),
            nn.GELU(),
            nn.Dropout(config["classifier_dropout"]),
            nn.Linear(ch, config["num_classes"]),
        )

    def forward(self, input_ids, attention_mask, token_type_ids, pixel_values):
        # Encode
        text_out   = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        text_feats = self.text_norm(text_out.last_hidden_state)
        img_feats  = self.image_norm(self.image_encoder(pixel_values))

        # Cross-attention
        text_mask = (attention_mask == 0)  # True = masked
        for layer in self.co_attn_layers:
            text_feats, img_feats = layer(text_feats, img_feats, text_key_padding_mask=text_mask)

        # Pool
        t_pool = self.text_pool(text_feats, mask=attention_mask.bool())
        v_pool = self.image_pool(img_feats)

        # Fuse: [t; v; |t-v|]
        fused = torch.cat([t_pool, v_pool, torch.abs(t_pool - v_pool)], dim=-1)
        fused = fused * self.fusion_gate(fused)  # gating

        return self.classifier(fused)


# Build model
device = torch.device(CONFIG["device"])
model  = ViLBERTOptimized(CONFIG).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
vision_params    = sum(p.numel() for p in model.image_encoder.backbone.parameters() if p.requires_grad)

print(f"Total parameters      : {total_params:,}")
print(f"Trainable parameters  : {trainable_params:,}")
print(f"Trainable vision      : {vision_params:,}")

# Quick sanity check
model.eval()
with torch.no_grad():
    _b = next(iter(train_loader))
    _logits = model(_b["input_ids"].to(device), _b["attention_mask"].to(device),
                    _b["token_type_ids"].to(device), _b["image"].to(device))
print(f"Forward pass OK  |  logits: {tuple(_logits.shape)} (expected [B, 3])")
del _b, _logits
model.train()
print(model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15308.24it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters      : 244,310,021
Trainable parameters  : 187,607,045
Trainable vision      : 29,095,680
Forward pass OK  |  logits: (16, 3) (expected [B, 3])
ViLBERTOptimized(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output):

In [9]:
# =====================================================
# Cell 8 - EMA Helper
# =====================================================

class EMA:
    """Exponential Moving Average of model parameters for better generalization."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model):
        """Replace model params with EMA params (for evaluation)."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        """Restore original model params (after evaluation)."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}

if CONFIG["use_ema"]:
    ema = EMA(model, decay=CONFIG["ema_decay"])
    print(f"EMA initialized with decay={CONFIG['ema_decay']}")
else:
    ema = None
    print("EMA disabled")

EMA initialized with decay=0.999


In [10]:
# =====================================================
# Cell 9 - Training Loop (Optimized)
# =====================================================

def move_batch(batch, device):
    return {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}


@torch.no_grad()
def evaluate(loader, model, criterion, device, use_amp=False):
    """Evaluate model on a dataloader. Returns dict with loss, acc, f1, preds, labels."""
    model.eval()
    total_loss, valid_batches = 0.0, 0
    all_preds, all_labels = [], []

    for batch in loader:
        batch = move_batch(batch, device)
        with torch.amp.autocast('cuda', enabled=use_amp and device.type == "cuda"):
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch["token_type_ids"], batch["image"])
            loss = criterion(logits, batch["label"])

        if torch.isfinite(loss):
            total_loss += loss.item()
            valid_batches += 1
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(batch["label"].cpu().tolist())

    avg_loss = total_loss / max(valid_batches, 1)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    f1w = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return {"loss": avg_loss, "acc": acc, "f1": f1, "f1w": f1w,
            "preds": all_preds, "labels": all_labels}


# ── Optimizer with differential LR ────────────────────────
bert_ids   = {id(p) for p in model.bert.parameters()}
vision_ids = {id(p) for p in model.image_encoder.backbone.parameters() if p.requires_grad}

bert_params   = [p for p in model.parameters() if id(p) in bert_ids]
vision_params = [p for p in model.parameters() if id(p) in vision_ids]
new_params    = [p for p in model.parameters()
                 if p.requires_grad and id(p) not in bert_ids and id(p) not in vision_ids]

optimizer = AdamW([
    {"params": bert_params,   "lr": CONFIG["bert_lr"]},
    {"params": vision_params, "lr": CONFIG["vision_lr"]},
    {"params": new_params,    "lr": CONFIG["new_params_lr"]},
], weight_decay=CONFIG["weight_decay"])

# ── Scheduler ─────────────────────────────────────────────
updates_per_epoch = math.ceil(len(train_loader) / CONFIG["grad_accum_steps"])
total_steps  = updates_per_epoch * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)

# ── Loss function ─────────────────────────────────────────
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(CONFIG["num_classes"]),
    y=df_train["label"].values,
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG["label_smoothing"],
)

scaler = torch.amp.GradScaler('cuda', enabled=CONFIG["use_amp"] and device.type == "cuda")

print(f"Class weights: neg={class_weights_np[0]:.3f}  neu={class_weights_np[1]:.3f}  pos={class_weights_np[2]:.3f}")
print(f"Optimizer: BERT={len(bert_params)}  Vision={len(vision_params)}  New={len(new_params)} tensors")
print(f"Scheduler: {total_steps} steps, {warmup_steps} warmup")

# ── Training ──────────────────────────────────────────────
best_val_f1      = 0.0
best_ckpt_path   = os.path.join(CONFIG["checkpoint_dir"], "vilbert_opt_best.pt")
patience_counter = 0
history          = []

header = f"{'Ep':>3}  {'TrLoss':>7}  {'TrAcc':>6}  {'VLoss':>7}  {'VAcc':>6}  {'VF1m':>6}  {'VF1w':>6}  {'LR':>9}  Status"
print(f"\n{header}")
print('-' * len(header))

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss = 0.0
    train_preds, train_labels = [], []
    epoch_start = time.time()

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch, device)

        with torch.amp.autocast('cuda', enabled=CONFIG["use_amp"] and device.type == "cuda"):
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch["token_type_ids"], batch["image"])
            loss = criterion(logits, batch["label"])

            # R-Drop: forward twice with dropout, minimize KL divergence
            if CONFIG["use_rdrop"]:
                logits2 = model(batch["input_ids"], batch["attention_mask"],
                                batch["token_type_ids"], batch["image"])
                loss2 = criterion(logits2, batch["label"])
                # KL divergence between two forward passes
                p = F.log_softmax(logits, dim=-1)
                q = F.log_softmax(logits2, dim=-1)
                kl = 0.5 * (F.kl_div(p, q.detach().exp(), reduction='batchmean') +
                            F.kl_div(q, p.detach().exp(), reduction='batchmean'))
                loss = 0.5 * (loss + loss2) + CONFIG["rdrop_alpha"] * kl

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss / CONFIG["grad_accum_steps"]).backward()

        should_step = (step % CONFIG["grad_accum_steps"] == 0) or (step == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            if ema is not None:
                ema.update(model)

        train_loss += loss.item()
        train_preds.extend(logits.detach().argmax(dim=-1).cpu().tolist())
        train_labels.extend(batch["label"].cpu().tolist())

    train_loss /= max(len(train_loader), 1)
    train_acc = accuracy_score(train_labels, train_preds)

    # Evaluate with EMA weights
    if ema is not None:
        ema.apply_shadow(model)

    val_metrics = evaluate(val_loader, model, criterion, device, use_amp=CONFIG["use_amp"])

    if ema is not None:
        ema.restore(model)

    lr_now = optimizer.param_groups[-1]["lr"]
    elapsed = time.time() - epoch_start

    # Early stopping logic
    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        patience_counter = 0
        status = "* BEST *"
        # Save best with EMA weights
        save_state = model.state_dict()
        if ema is not None:
            ema.apply_shadow(model)
            save_state = copy.deepcopy(model.state_dict())
            ema.restore(model)
        torch.save({
            "epoch": epoch, "model_state": save_state,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "val_metrics": val_metrics, "config": copy.deepcopy(CONFIG),
        }, best_ckpt_path)
    else:
        patience_counter += 1
        status = f"p={patience_counter}/{CONFIG['early_stop_patience']}"

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_metrics["loss"], "val_acc": val_metrics["acc"],
        "val_f1": val_metrics["f1"], "val_f1w": val_metrics["f1w"],
        "lr": lr_now, "time_s": elapsed,
    })

    print(
        f"{epoch:>3}  {train_loss:>7.4f}  {train_acc:>6.4f}  {val_metrics['loss']:>7.4f}  "
        f"{val_metrics['acc']:>6.4f}  {val_metrics['f1']:>6.4f}  {val_metrics['f1w']:>6.4f}  "
        f"{lr_now:>9.2e}  {status}  ({elapsed:.0f}s)"
    )

    if patience_counter >= CONFIG["early_stop_patience"]:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {CONFIG['early_stop_patience']} epochs)")
        break

print(f"\nTraining complete. Best val macro-F1: {best_val_f1:.4f}")
print(f"Best checkpoint: {best_ckpt_path}")

Class weights: neg=1.107  neu=3.199  pos=0.560
Optimizer: BERT=199  Vision=54  New=94 tensors
Scheduler: 3390 steps, 339 warmup

 Ep   TrLoss   TrAcc    VLoss    VAcc    VF1m    VF1w         LR  Status
------------------------------------------------------------------------
  1   1.0978  0.4900   1.1551  0.3858  0.3106  0.4094   3.33e-05  * BEST *  (1861s)
  2   0.8800  0.6789   1.1063  0.4989  0.4102  0.5189   6.67e-05  * BEST *  (2181s)
  3   0.7157  0.7625   1.0514  0.5787  0.4992  0.5923   1.00e-04  * BEST *  (1893s)
  4   0.5673  0.8550   0.9970  0.6297  0.5615  0.6383   9.97e-05  * BEST *  (1743s)
  5   0.4252  0.9358   0.9479  0.6608  0.5979  0.6676   9.87e-05  * BEST *  (1743s)
  6   0.3885  0.9547   0.9134  0.6940  0.6346  0.6982   9.70e-05  * BEST *  (1744s)
  7   0.3515  0.9700   0.8951  0.7073  0.6415  0.7092   9.47e-05  * BEST *  (1744s)
  8   0.3346  0.9808   0.8921  0.7118  0.6442  0.7118   9.18e-05  * BEST *  (1744s)
  9   0.3228  0.9836   0.9019  0.7273  0.6626  0.7267

In [11]:
# =====================================================
# Cell 10 - Test Set Evaluation
# =====================================================

# Load best checkpoint
print(f"Loading best checkpoint from: {best_ckpt_path}")
checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
print(f"Checkpoint epoch: {checkpoint['epoch']}  val_f1={checkpoint['val_metrics']['f1']:.4f}")

# Evaluate on test set
test_metrics = evaluate(test_loader, model, criterion, device, use_amp=CONFIG["use_amp"])

class_names = [LABEL_IMAP[i] for i in range(CONFIG["num_classes"])]
cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])

SEP = '=' * 64
print(f"\n{SEP}")
print(f"  TEST SET RESULTS  -  ViLBERT-Optimized")
print(SEP)
print(f"  Accuracy         : {test_metrics['acc']:.4f}  ({test_metrics['acc']*100:.2f}%)")
print(f"  Macro F1         : {test_metrics['f1']:.4f}")
print(f"  Weighted F1      : {test_metrics['f1w']:.4f}")
print(f"  Best val macro-F1: {best_val_f1:.4f}")

print(f"\n  Classification Report:")
print(classification_report(
    test_metrics["labels"], test_metrics["preds"],
    target_names=class_names, digits=4, zero_division=0,
))

print(f"  Confusion Matrix (rows=true, cols=pred):")
header_str = "           " + "  ".join(f"{n:>10}" for n in class_names)
print(header_str)
for idx, row in enumerate(cm):
    row_str = "  ".join(f"{v:>10}" for v in row)
    print(f"  {class_names[idx]:>8}  {row_str}")

# Training history
if history:
    h_df = pd.DataFrame(history)
    print(f"\n  Training History (last 10 epochs):")
    print(h_df.tail(10).to_string(index=False))

print(f"\n{SEP}")

Loading best checkpoint from: D:\/MVSA_SINGLE/vilbert_opt_checkpoints\vilbert_opt_best.pt
Checkpoint epoch: 10  val_f1=0.6903

  TEST SET RESULTS  -  ViLBERT-Optimized
  Accuracy         : 0.7279  (72.79%)
  Macro F1         : 0.6312
  Weighted F1      : 0.7249
  Best val macro-F1: 0.6903

  Classification Report:
              precision    recall  f1-score   support

    negative     0.7190    0.6397    0.6770       136
     neutral     0.4222    0.4043    0.4130        47
    positive     0.7797    0.8290    0.8036       269

    accuracy                         0.7279       452
   macro avg     0.6403    0.6243    0.6312       452
weighted avg     0.7243    0.7279    0.7249       452

  Confusion Matrix (rows=true, cols=pred):
             negative     neutral    positive
  negative          87           8          41
   neutral           6          19          22
  positive          28          18         223

  Training History (last 10 epochs):
 epoch  train_loss  train_acc  val_

In [12]:
# =====================================================
# Cell 11 - Save Model & Artifacts
# =====================================================

SAVE_DIR = CONFIG["save_dir"]
os.makedirs(SAVE_DIR, exist_ok=True)

full_ckpt_path = os.path.join(SAVE_DIR, "vilbert_opt_full.pt")
weights_path   = os.path.join(SAVE_DIR, "vilbert_opt_weights.pt")
config_path    = os.path.join(SAVE_DIR, "vilbert_opt_config.json")
history_path   = os.path.join(SAVE_DIR, "vilbert_opt_history.csv")
summary_path   = os.path.join(SAVE_DIR, "vilbert_opt_summary.json")

# Full checkpoint
torch.save({
    "model_state_dict"     : model.state_dict(),
    "optimizer_state_dict" : optimizer.state_dict(),
    "config"               : CONFIG,
    "label_map"            : LABEL_MAP,
    "label_imap"           : LABEL_IMAP,
    "best_val_f1"          : best_val_f1,
    "test_acc"             : test_metrics["acc"],
    "test_f1"              : test_metrics["f1"],
    "test_f1w"             : test_metrics["f1w"],
    "history"              : history,
}, full_ckpt_path)
print(f"Full checkpoint  : {full_ckpt_path}")

# Weights only
torch.save(model.state_dict(), weights_path)
print(f"Weights-only     : {weights_path}")

# Config JSON
cfg_save = {k: v for k, v in CONFIG.items() if not callable(v)}
cfg_save["label_map"]  = LABEL_MAP
cfg_save["label_imap"] = {str(k): v for k, v in LABEL_IMAP.items()}
with open(config_path, "w") as f:
    json.dump(cfg_save, f, indent=2)
print(f"Config JSON      : {config_path}")

# History CSV
pd.DataFrame(history).to_csv(history_path, index=False)
print(f"History CSV      : {history_path}")

# Summary JSON
with open(summary_path, "w") as f:
    json.dump({
        "model": "vilbert_optimized",
        "best_val_f1": best_val_f1,
        "test_acc": test_metrics["acc"],
        "test_f1": test_metrics["f1"],
        "test_f1w": test_metrics["f1w"],
        "epochs_trained": len(history),
        "checkpoint": best_ckpt_path,
    }, f, indent=2)
print(f"Summary JSON     : {summary_path}")

# Print file sizes
print(f"\nSaved files:")
for p in [full_ckpt_path, weights_path, config_path, history_path, summary_path]:
    mb = os.path.getsize(p) / (1024**2)
    print(f"  {os.path.basename(p):<35}  {mb:>8.2f} MB")

print(f"\n{'=' * 64}")
print(f"  RELOAD INSTRUCTIONS")
print(f"{'=' * 64}")
print(f"""
  import json, torch
  from transformers import BertTokenizer

  with open("{os.path.basename(config_path)}") as f:
      cfg = json.load(f)

  model = ViLBERTOptimized(cfg)
  ckpt = torch.load("{os.path.basename(full_ckpt_path)}", map_location="cpu")
  model.load_state_dict(ckpt["model_state_dict"])
  model.eval()

  tokenizer = BertTokenizer.from_pretrained(cfg["bert_model_name"])
  # logits = model(input_ids, attention_mask, token_type_ids, pixel_values)
  # pred = logits.argmax(dim=-1).item()
""")

Full checkpoint  : D:\/MVSA_SINGLE/vilbert_opt_final\vilbert_opt_full.pt
Weights-only     : D:\/MVSA_SINGLE/vilbert_opt_final\vilbert_opt_weights.pt
Config JSON      : D:\/MVSA_SINGLE/vilbert_opt_final\vilbert_opt_config.json
History CSV      : D:\/MVSA_SINGLE/vilbert_opt_final\vilbert_opt_history.csv
Summary JSON     : D:\/MVSA_SINGLE/vilbert_opt_final\vilbert_opt_summary.json

Saved files:
  vilbert_opt_full.pt                   2359.26 MB
  vilbert_opt_weights.pt                 932.15 MB
  vilbert_opt_config.json                  0.00 MB
  vilbert_opt_history.csv                  0.00 MB
  vilbert_opt_summary.json                 0.00 MB

  RELOAD INSTRUCTIONS

  import json, torch
  from transformers import BertTokenizer

  with open("vilbert_opt_config.json") as f:
      cfg = json.load(f)

  model = ViLBERTOptimized(cfg)
  ckpt = torch.load("vilbert_opt_full.pt", map_location="cpu")
  model.load_state_dict(ckpt["model_state_dict"])
  model.eval()

  tokenizer = BertTokenizer.fro